## RAG pipeline

#### Data ingestion + chunking

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import pandas as pd

import os
from pathlib import Path

# The below code is used to access packages from the root
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

from app.agent.rag.ingestion.data_ingestion import process_and_load_file
from app.agent.rag.ingestion.embeddings import get_embedding_model, generate_embeddings

In [3]:
print(os.getcwd())

c:\Users\USER\Documents\AI projects\DenseLess\app\agent


In [4]:
docs = process_and_load_file("../pdfs/Software Engineering Intro.pdf")
print(docs)

INFO: split_pdf event=plan_created operation_id=e88280cf-ea2a-4f12-afdb-ad7dd3e6dabd filename=Software Engineering Intro.pdf strategy=hi_res page_range=1-5 page_count=5 split_size=2 chunk_count=3 concurrency=5 allow_failed=False cache_mode=disabled timeout_seconds=None retry_config_mode=sdk_default_or_unset


PDF page count: 5
Processing: Software Engineering Intro.pdf


INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_start operation_id=e88280cf-ea2a-4f12-afdb-ad7dd3e6dabd chunk_count=3 concurrency=5 allow_failed=False client_timeout_seconds=None future_timeout_seconds=3605 num_waves=1
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_complete operation_id=e88280cf-ea2a-4f12-afdb-ad7dd3e6dabd chunk_count=3 success_count=3 failure_count=0 transport_failure_count=0 elapsed_ms=9732 allow_failed=False


  → Parsed into 49 document(s)
[Document(metadata={'section': 'Course Contents', 'category': 'Title', 'page_number': 1, 'element_id': '330cf4934ef83395d3354a751d440c5e', 'filename': 'Software Engineering Intro.pdf'}, page_content='Course Contents:'), Document(metadata={'section': 'Course Contents', 'category': 'NarrativeText', 'page_number': 1, 'element_id': '178ec6be141992c8bbf48e59124c3669', 'filename': 'Software Engineering Intro.pdf'}, page_content='Introduction; Software Design: Software architecture, Design patterns, OO analysis & Design, Design for re-use. Using API’s: API programming, Class browsers and Related tools, Component based computing. Software tools and Environment: Requirements analysis and design modeling tools, Testing tools, Tool integration mechanisms.'), Document(metadata={'section': 'The Term Software', 'category': 'Title', 'page_number': 1, 'element_id': '2f2e528cd35f89c8819797c1823e8303', 'filename': 'Software Engineering Intro.pdf'}, page_content='The Term S

In [4]:
from langchain_core.documents import Document

docs = [Document(metadata={'section': 'Course Contents', 'category': 'Title', 'page_number': 1, 'element_id': '330cf4934ef83395d3354a751d440c5e', 'filename': 'Software Engineering Intro.pdf'}, page_content='Course Contents:'), Document(metadata={'section': 'Course Contents', 'category': 'NarrativeText', 'page_number': 1, 'element_id': '178ec6be141992c8bbf48e59124c3669', 'filename': 'Software Engineering Intro.pdf'}, page_content='Introduction; Software Design: Software architecture, Design patterns, OO analysis & Design, Design for re-use. Using API’s: API programming, Class browsers and Related tools, Component based computing. Software tools and Environment: Requirements analysis and design modeling tools, Testing tools, Tool integration mechanisms.'), Document(metadata={'section': 'The Term Software', 'category': 'Title', 'page_number': 1, 'element_id': '2f2e528cd35f89c8819797c1823e8303', 'filename': 'Software Engineering Intro.pdf'}, page_content='The Term Software'), Document(metadata={'section': 'The Term Software', 'category': 'NarrativeText', 'page_number': 1, 'element_id': '6fd8fca74c7e754c6d08fcc720fbe65f', 'filename': 'Software Engineering Intro.pdf'}, page_content='Software is a set of instructions used to acquire inputs and to manipulate them to produce the desired output in terms of functions and performance as determined by the user of the software. It also includes a set of documents, such as the software manual, meant to help users understand the software system.'), Document(metadata={'section': 'Today’s software is comprised of', 'category': 'Title', 'page_number': 1, 'element_id': '17f3ae59ea08c812322cc15c1a19e422', 'filename': 'Software Engineering Intro.pdf'}, page_content='Today’s software is comprised of:'), Document(metadata={'section': 'Today’s software is comprised of', 'category': 'ListItem', 'page_number': 1, 'element_id': '67e211ff675dc10c9263bdbbb3c31cb8', 'filename': 'Software Engineering Intro.pdf'}, page_content='1. Source Code: These are programs or codes written by the programmers using any particular programming language such as Java, VB, Python, etc.'), Document(metadata={'section': 'Today’s software is comprised of', 'category': 'ListItem', 'page_number': 1, 'element_id': '759eebbe2824ba9e9a523b2da6288819', 'filename': 'Software Engineering Intro.pdf'}, page_content='2. Executable: These are compiled codes that can run without the source code and can be shared with other users to run independent of the source program.'), Document(metadata={'section': 'Today’s software is comprised of', 'category': 'ListItem', 'page_number': 1, 'element_id': 'dcef89134c18fbf4ab3ebbde20de2263', 'filename': 'Software Engineering Intro.pdf'}, page_content='3. Design Documents: These are different designs necessary when trying to develop software such as input design, output design, UML, use case design, class diagrams, software architecture, etc.'), Document(metadata={'section': 'Today’s software is comprised of', 'category': 'ListItem', 'page_number': 1, 'element_id': '72cbed0c6e229d1fb721b3c077cf27fb', 'filename': 'Software Engineering Intro.pdf'}, page_content='4. Operations: These are the functions expected of any software to be developed for any purpose by the software engineer of the developer. Functions can be listed to guide the developer on the way forward on each module of the software. It may be programmer centred, generic, or customized, etc.'), Document(metadata={'section': 'Today’s software is comprised of', 'category': 'ListItem', 'page_number': 1, 'element_id': 'e95c20ae1a3f6dbc9a6088c176e0dc1f', 'filename': 'Software Engineering Intro.pdf'}, page_content='5. System Manuals: Manuals are very important for the users, there is need for user’s guide or user manual, technical manual, complete system manual, etc.'), Document(metadata={'section': 'Today’s software is comprised of', 'category': 'ListItem', 'page_number': 2, 'element_id': '9848358fc7b7184da9fc857f0177f9b4', 'filename': 'Software Engineering Intro.pdf'}, page_content='6. Installation: After the development of the software by the developer or programmer, it must be installed on the user’s computer system. The process of making it function form the user’s system is call installation which depends on the hardware and the operating system.'), Document(metadata={'section': 'Today’s software is comprised of', 'category': 'ListItem', 'page_number': 2, 'element_id': 'f02ce49041b818e24bea633b5a85de40', 'filename': 'Software Engineering Intro.pdf'}, page_content='7. Implementation: After Installation, the use of the software for real life job by the organization is very important not to affect the existing system. This can be by any technique such as direct, pilot, phased or parallel technique depending on the type of the organisation, the software and the use.'), Document(metadata={'section': 'Today’s software is comprised of', 'category': 'NarrativeText', 'page_number': 2, 'element_id': 'db410f6ce2405fc0043255f8397a4b59', 'filename': 'Software Engineering Intro.pdf'}, page_content='Software also includes:'), Document(metadata={'section': 'Today’s software is comprised of', 'category': 'ListItem', 'page_number': 2, 'element_id': '15457b4a56350bb8213e95623369e129', 'filename': 'Software Engineering Intro.pdf'}, page_content='i. Instructions (computer programs) that when executed provide desired functions and performance.'), Document(metadata={'section': 'Today’s software is comprised of', 'category': 'ListItem', 'page_number': 2, 'element_id': 'd0c09d8ccadcccc9022a3afc4be50342', 'filename': 'Software Engineering Intro.pdf'}, page_content='ii. Data structures that enable the programs to adequately manipulate information.'), Document(metadata={'section': 'Today’s software is comprised of', 'category': 'ListItem', 'page_number': 2, 'element_id': '67410d6bfa3057f8f473bec7d8d515d6', 'filename': 'Software Engineering Intro.pdf'}, page_content='iii. Documents that describe the operation and use to dispatch the goods. The facilities and features could be optional and based on customer choices.'), Document(metadata={'section': 'Today’s software is comprised of', 'category': 'NarrativeText', 'page_number': 2, 'element_id': '5fe16bcddb3555435f315609d48431cb', 'filename': 'Software Engineering Intro.pdf'}, page_content='Software is developed keeping in mind certain hardware and operating system considerations, known as the platform. Hence, software is described along with its capabilities and the platform specifications that are required to run it.'), Document(metadata={'section': 'Note: Practical on how to differentiate between source code and executable USING Visual Basic', 'category': 'Title', 'page_number': 2, 'element_id': 'd3259d53de7217c87a471998d5acd149', 'filename': 'Software Engineering Intro.pdf'}, page_content='Note: Practical on how to differentiate between source code and executable USING Visual Basic'), Document(metadata={'section': 'The Term Engineering', 'category': 'Title', 'page_number': 2, 'element_id': 'd0974e598bddf135ff56d9d98cfac5b6', 'filename': 'Software Engineering Intro.pdf'}, page_content='The Term Engineering'), Document(metadata={'section': 'The Term Engineering', 'category': 'NarrativeText', 'page_number': 2, 'element_id': '32d358d171dcabdd8c7e050a16b20e63', 'filename': 'Software Engineering Intro.pdf'}, page_content='Engineering generally means to make, achieve or get through contrivance or guile while contrivance means something, as a machine, devised for a particular function, something invented and specialised mechanical devices. Guile in its own case means deceit which is related to finding solution to problems that are not related to machines but cunning in nature or deceived in nature. More than a discipline or a body of knowledge, engineering is a verb, an action word, a way of approaching a problem. (Roger, 2005).'), Document(metadata={'section': 'The Term Engineering', 'category': 'NarrativeText', 'page_number': 3, 'element_id': '1e88c356cffca5b6ed3088c3606d902a', 'filename': 'Software Engineering Intro.pdf'}, page_content='Adam et al (2001) stated that engineering means the profession and activity of designing the way, roads, bridges, machines etc. or machinery and equipment used or developed as a result of knowledge.'), Document(metadata={'section': 'The Term Engineering', 'category': 'NarrativeText', 'page_number': 3, 'element_id': '34b1ac2cc7393b56b2ca384ca6c48b72', 'filename': 'Software Engineering Intro.pdf'}, page_content='Source: Moore (1998). Software Engineering Standards, IEEE, 1998'), Document(metadata={'section': 'The Term Engineering', 'category': 'NarrativeText', 'page_number': 3, 'element_id': '08b8fa696c9b063dd9b63370a06adc0a', 'filename': 'Software Engineering Intro.pdf'}, page_content='The model above shows that, for any engineering process, there must be goals to be achieved while constraints will be associated with it which may likely affect one way or the other. Engineer is to control everything using measurement, action and process by getting the needs and necessary resources which will help in achieving the goals to produce the product or provide the services. This is a complete definition of engineering.'), Document(metadata={'section': 'The Term Software Engineering', 'category': 'Title', 'page_number': 3, 'element_id': 'a5868ab2eabb680c691e2b439ccc535e', 'filename': 'Software Engineering Intro.pdf'}, page_content='The Term Software Engineering'), Document(metadata={'section': 'The Term Software Engineering', 'category': 'NarrativeText', 'page_number': 3, 'element_id': 'ef519e6f31702715ba5bcc6abb89f3ac', 'filename': 'Software Engineering Intro.pdf'}, page_content='Software Engineering is commonly used for larger and complex software systems. It can be referred to as the application of engineering principles in software development. The emphasis on software engineering in computing is on how to achieve best practices and engineering processes that can be applied to software development.'), Document(metadata={'section': 'The Term Software Engineering', 'category': 'NarrativeText', 'page_number': 4, 'element_id': '4cee9542dafb578078be7c30f62e7447', 'filename': 'Software Engineering Intro.pdf'}, page_content='Software engineering is the process of analyzing user needs and designing, constructing, and testing end user applications that will satisfy these needs through the use of software programming languages. It is the application of engineering principles to software development.'), Document(metadata={'section': 'The Term Software Engineering', 'category': 'NarrativeText', 'page_number': 4, 'element_id': '0e6931587bb2ef5c176b0fbdaf475135', 'filename': 'Software Engineering Intro.pdf'}, page_content='A few other important definitions given by several authors and institutions are as follows:'), Document(metadata={'section': 'The Term Software Engineering', 'category': 'NarrativeText', 'page_number': 4, 'element_id': 'eb296d44bc926c1a9545fc072b4bb62b', 'filename': 'Software Engineering Intro.pdf'}, page_content='IEEE Comprehensive Definition. Software Engineering is the application of a systematic, disciplined, quantifiable approach to the development, operation and maintenance of software, i.e., the application of engineering to software.'), Document(metadata={'section': 'The Term Software Engineering', 'category': 'NarrativeText', 'page_number': 4, 'element_id': '570ac9849b370db2f3c81f9a7c74c0f4', 'filename': 'Software Engineering Intro.pdf'}, page_content='Software Engineering deals with cost-effective solutions to practical problems by applying scientific knowledge in building software artifacts in the service of mankind.'), Document(metadata={'section': 'OR', 'category': 'NarrativeText', 'page_number': 4, 'element_id': '404369ec43aabffd16b27c4bac0cdd94', 'filename': 'Software Engineering Intro.pdf'}, page_content='Software Engineering is the application of methods and scientific knowledge to create practical cost-effective solutions for the design, construction, operation and maintenance of software.'), Document(metadata={'section': 'OR', 'category': 'NarrativeText', 'page_number': 4, 'element_id': '108716cf474d0dbcd3f847b9e3ddeaff', 'filename': 'Software Engineering Intro.pdf'}, page_content='Software Engineering is a discipline whose aim is the production of fault free software that satisfies the user’s needs and that is delivered on time and within budget.'), Document(metadata={'section': 'OR', 'category': 'NarrativeText', 'page_number': 4, 'element_id': '11708d6c1089a59826e8e94dcbd8d31d', 'filename': 'Software Engineering Intro.pdf'}, page_content='The term Software Engineering refers to a movement, methods and techniques aimed at making software development more systematic.'), Document(metadata={'section': 'Computer Science VS Software Engineering', 'category': 'Title', 'page_number': 4, 'element_id': 'ea5ba6a5b13e0f872a364ee6d882d702', 'filename': 'Software Engineering Intro.pdf'}, page_content='Computer Science VS Software Engineering'), Document(metadata={'section': 'Computer Science VS Software Engineering', 'category': 'ListItem', 'page_number': 4, 'element_id': '6b235bfd645d043a2eb4ceda687d5222', 'filename': 'Software Engineering Intro.pdf'}, page_content='Computer Science is the study of how computers work, mostly from the theoretical and mathematical perspective.'), Document(metadata={'section': 'Computer Science VS Software Engineering', 'category': 'ListItem', 'page_number': 4, 'element_id': '4698521139b45238c52d954f78606630', 'filename': 'Software Engineering Intro.pdf'}, page_content='You should choose Computer Science if you like math, logic, or if you want to get into a specialized field in Computer Science such as artificial intelligence, machine learning, security, or graphics.'), Document(metadata={'section': 'Computer Science VS Software Engineering', 'category': 'ListItem', 'page_number': 5, 'element_id': '13bf96bca463f16ed6827b4be91d1678', 'filename': 'Software Engineering Intro.pdf'}, page_content='Software Engineering is the study of how software systems are built, including topics such as project management, software architecture, risk management, quality assurance, software testing, etc.'), Document(metadata={'section': 'Computer Science VS Software Engineering', 'category': 'ListItem', 'page_number': 5, 'element_id': 'd4caaba16eb5cd503f989cbe977f9556', 'filename': 'Software Engineering Intro.pdf'}, page_content='You should choose Software Engineering if you are more interested in the hands-on approach, and if you want to learn the overall life cycle of how software is built and maintained.'), Document(metadata={'section': 'Computer Science VS Software Engineering', 'category': 'ListItem', 'page_number': 5, 'element_id': 'a977347c3d0f05922e099dc6a0ac321b', 'filename': 'Software Engineering Intro.pdf'}, page_content='Both Computer Science and Software Engineering teach fundamentals of programming and computer science, so you can choose either one to become a software developer.'), Document(metadata={'section': 'Computer Science VS Software Engineering', 'category': 'ListItem', 'page_number': 5, 'element_id': '621e50a9f54909537fe87a09852c3513', 'filename': 'Software Engineering Intro.pdf'}, page_content='The core computer science requirements are similar as well, ranging over algorithms, data structures, and operating systems.'), Document(metadata={'section': 'The key differences are', 'category': 'Title', 'page_number': 5, 'element_id': '9b2a678879b6124327e4ec3fd9c7b9f7', 'filename': 'Software Engineering Intro.pdf'}, page_content='The key differences are:'), Document(metadata={'section': 'The key differences are', 'category': 'ListItem', 'page_number': 5, 'element_id': 'f38d9b190cd3f9e585c9998d12121425', 'filename': 'Software Engineering Intro.pdf'}, page_content='Software Engineering has more requirements in software engineering fundamentals, such as software testing, design, and software requirements specification.'), Document(metadata={'section': 'The key differences are', 'category': 'ListItem', 'page_number': 5, 'element_id': 'b07b88fed6eedf1f029983c595c724bc', 'filename': 'Software Engineering Intro.pdf'}, page_content='Computer Science allows more electives in higher-level computer science courses. You can choose from a wide range of topics such as security, software engineering fundamentals, computer vision, machine learning, and database management.'), Document(metadata={'section': 'Importance of Software', 'category': 'Title', 'page_number': 5, 'element_id': 'dc0443f652e4362b391e25ce02bbfff6', 'filename': 'Software Engineering Intro.pdf'}, page_content='Importance of Software'), Document(metadata={'section': 'Importance of Software', 'category': 'NarrativeText', 'page_number': 5, 'element_id': '7a34131b63dd2ee29078601db24684b5', 'filename': 'Software Engineering Intro.pdf'}, page_content='Computer software has become a driving force.'), Document(metadata={'section': 'Importance of Software', 'category': 'ListItem', 'page_number': 5, 'element_id': '0da36b9c627500b9d8e55d06bef9fbbd', 'filename': 'Software Engineering Intro.pdf'}, page_content='1. It is the engine that drives business decision making.'), Document(metadata={'section': 'Importance of Software', 'category': 'ListItem', 'page_number': 5, 'element_id': '543b135107dd93f24d2c521124b5fd1b', 'filename': 'Software Engineering Intro.pdf'}, page_content='2. It serves as the basis for modern scientific investigation and engineering problem-solving.'), Document(metadata={'section': 'Importance of Software', 'category': 'ListItem', 'page_number': 5, 'element_id': '8fa6932656f61c45aac66e5ac1a09a46', 'filename': 'Software Engineering Intro.pdf'}, page_content='3. It is embedded in all kinds of systems, such as transportation, medical, telecommunications, military, industrial processes, entertainment, office products, etc.'), Document(metadata={'section': 'Importance of Software', 'category': 'ListItem', 'page_number': 5, 'element_id': 'b6253f21057f623b1a0d06d189253970', 'filename': 'Software Engineering Intro.pdf'}, page_content='4. It is important as it affects nearly every aspect of our lives and has becomepervasive in our commerce, our culture, and our everyday activities. Software’simpact on our society and culture is significant.'), Document(metadata={'section': 'Importance of Software', 'category': 'NarrativeText', 'page_number': 5, 'element_id': 'c9eaf82d26d6919f87f892a8f01bfa71', 'filename': 'Software Engineering Intro.pdf'}, page_content='As software importance grows, thesoftware community continually attempts to develop technologies that will makeit easier, faster, and less expensive to build high-quality computer programs.')]
print(docs)

[Document(metadata={'section': 'Course Contents', 'category': 'Title', 'page_number': 1, 'element_id': '330cf4934ef83395d3354a751d440c5e', 'filename': 'Software Engineering Intro.pdf'}, page_content='Course Contents:'), Document(metadata={'section': 'Course Contents', 'category': 'NarrativeText', 'page_number': 1, 'element_id': '178ec6be141992c8bbf48e59124c3669', 'filename': 'Software Engineering Intro.pdf'}, page_content='Introduction; Software Design: Software architecture, Design patterns, OO analysis & Design, Design for re-use. Using API’s: API programming, Class browsers and Related tools, Component based computing. Software tools and Environment: Requirements analysis and design modeling tools, Testing tools, Tool integration mechanisms.'), Document(metadata={'section': 'The Term Software', 'category': 'Title', 'page_number': 1, 'element_id': '2f2e528cd35f89c8819797c1823e8303', 'filename': 'Software Engineering Intro.pdf'}, page_content='The Term Software'), Document(metadata={'

In [22]:
pd.set_option('display.max_rows', None)

df = pd.DataFrame([
    {**doc.metadata, "content_preview": doc.page_content[:300], "character count": len(doc.page_content)}
    for doc in docs
])
# df

In [23]:
df

,section,category,page_number,element_id,filename,content_preview,character count
0,Course Contents,Title,1,330cf4934ef83395d3354a751d440c5e,Software Engineering Intro.pdf,Course Contents:,16
1,Course Contents,NarrativeText,1,178ec6be141992c8bbf48e59124c3669,Software Engineering Intro.pdf,Introduction; Software Design: Software archit...,327
2,The Term Software,Title,1,2f2e528cd35f89c8819797c1823e8303,Software Engineering Intro.pdf,The Term Software,17
3,The Term Software,NarrativeText,1,6fd8fca74c7e754c6d08fcc720fbe65f,Software Engineering Intro.pdf,Software is a set of instructions used to acqu...,308
4,Today’s software is comprised of,Title,1,17f3ae59ea08c812322cc15c1a19e422,Software Engineering Intro.pdf,Today’s software is comprised of:,33
5,Today’s software is comprised of,ListItem,1,67e211ff675dc10c9263bdbbb3c31cb8,Software Engineering Intro.pdf,1. Source Code: These are programs or codes wr...,143
6,Today’s software is comprised of,ListItem,1,759eebbe2824ba9e9a523b2da6288819,Software Engineering Intro.pdf,2. Executable: These are compiled codes that c...,153
7,Today’s software is comprised of,ListItem,1,dcef89134c18fbf4ab3ebbde20de2263,Software Engineering Intro.pdf,3. Design Documents: These are different desig...,193
8,Today’s software is comprised of,ListItem,1,72cbed0c6e229d1fb721b3c077cf27fb,Software Engineering Intro.pdf,4. Operations: These are the functions expecte...,295
9,Today’s software is comprised of,ListItem,1,e95c20ae1a3f6dbc9a6088c176e0dc1f,Software Engineering Intro.pdf,5. System Manuals: Manuals are very important ...,154


In [ ]:
pd.set_option('display.max_rows', 40)
print(df)

                             section       category  page_number  \
0                    Course Contents          Title            1   
1                    Course Contents  NarrativeText            1   
2                  The Term Software          Title            1   
3                  The Term Software  NarrativeText            1   
4   Today’s software is comprised of          Title            1   
..                               ...            ...          ...   
44            Importance of Software       ListItem            5   
45            Importance of Software       ListItem            5   
46            Importance of Software       ListItem            5   
47            Importance of Software       ListItem            5   
48            Importance of Software  NarrativeText            5   

                          element_id                        filename  \
0   330cf4934ef83395d3354a751d440c5e  Software Engineering Intro.pdf   
1   178ec6be141992c8bbf48e59124c3669  S

In [ ]:
len(docs[11].page_content)

303

#### Embeddings

In [10]:
embedder = get_embedding_model()
vectors = generate_embeddings(docs, embedder)
print(vectors)

# import numpy as np
# np.save('SE_vectors.npy', vectors)

Loading embedding model: BAAI/bge-small-en-v1.5 (device=cpu)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  → Embedding model loaded successfully. Embedding dimension: 384
Generating embeddings for 49 document chunk(s)...


Embedding documents: 100%|██████████| 2/2 [00:01<00:00,  1.89it/s]

  → Successfully generated 49 embedding vectors.
  → Embeddings shape: (49, 384)
[[-0.00815396 -0.00902947  0.02052058 ...  0.00606262  0.08444569
   0.02088505]
 [-0.04133336 -0.02045064  0.01076466 ...  0.02804156  0.07489467
  -0.00789173]
 [-0.01306459 -0.03788544  0.02634777 ...  0.01196934  0.0561258
   0.02619192]
 ...
 [-0.01421627  0.00594315 -0.00272114 ... -0.03974212  0.09942809
   0.06926321]
 [ 0.03426296  0.00974565  0.02174552 ... -0.05176656  0.0728165
   0.04668332]
 [-0.0082825  -0.01495424  0.01532776 ... -0.03255153  0.11740864
   0.05944892]]


In [6]:
# OFFLINE USE
from langchain_huggingface import HuggingFaceEmbeddings


embedder = HuggingFaceEmbeddings(model_name='BAAI/bge-small-en-v1.5', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True}, query_encode_kwargs={}, multi_process=False, show_progress=False)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# OFFLINE USE 
import numpy as np

vectors = np.load('SE_vectors.npy')

In [8]:
from app.agent.rag.ingestion.vector_store import get_vector_store, add_documents_to_chroma

In [9]:
# This is my id
store = get_vector_store(student_id="1019", embedder=embedder)

Vector store backend: 'chroma' | student: '1019'
Initialising Chroma store — student: '1019'
  → Collection : student_1019
  → Persist dir: ../chroma_db
  → Chroma store ready. 
Documents in collection: 49


In [10]:
add_documents_to_chroma(store, vectors, docs, False, "CSC 403", "Software Engineering Intro", "Software Engineering Intro")

  [vector_store] 49 existing chunk(s) found in collection.
  [vector_store] 49 duplicate chunk(s) skipped.
  [vector_store] ✓ All documents already exist in store — nothing to add.


### Retrieval pipeline

In [11]:
from app.agent.rag.retrieval.retriever import get_semantic_chunks, get_topic_chunks
from IPython.display import display, Markdown

In [15]:
# Get relevant chunks through Q&A
chunks = get_semantic_chunks(query="What is Software", store=store, score_threshold=0.4, top_k=5, topic="Software Engineering Intro", course="CSC 403", condensed=False)
chunks

[Retriever] Phase 1 — Semantic search (top_k=5, threshold=0.4): 'What is Software'
Querying Chroma (top_k=5, threshold=0.4): 'Represent this sentence for searching relevant passages: Wha...'
  → Retrieved 5 chunk(s), 5 passed threshold.
[Retriever] → 5 seed chunk(s) passed threshold.
[Retriever] Phase 2 — Section context expansion...
[Retriever] Expanding context across 3 section(s): {'The Term Software', 'Today’s software is comprised of', 'OR'}
[Retriever]   → Section 'The Term Software': 2 chunk(s) fetched.
[Retriever]   → Section 'Today’s software is comprised of': 13 chunk(s) fetched.
[Retriever]   → Section 'OR': 3 chunk(s) fetched.
[Retriever] → Expansion complete: 5 seed → 18 unique chunk(s) after deduplication.
[Retriever] → Final result: 18 chunk(s) returned to chain.


[Document(id='doc_87f41029_2', metadata={'topic': 'Software Engineering Intro', 'source': 'CSC 403_Software Engineering Intro_Software Engineering Intro', 'filename': 'Software Engineering Intro.pdf', 'course': 'CSC 403', 'section': 'The Term Software', 'element_id': '2f2e528cd35f89c8819797c1823e8303', 'page_number': 1, 'condensed': False, 'doc_index': 2, 'category': 'Title', 'content_length': 17}, page_content='The Term Software'),
 Document(id='doc_94c56640_3', metadata={'source': 'CSC 403_Software Engineering Intro_Software Engineering Intro', 'filename': 'Software Engineering Intro.pdf', 'course': 'CSC 403', 'section': 'The Term Software', 'category': 'NarrativeText', 'page_number': 1, 'topic': 'Software Engineering Intro', 'content_length': 308, 'condensed': False, 'doc_index': 3, 'element_id': '6fd8fca74c7e754c6d08fcc720fbe65f'}, page_content='Software is a set of instructions used to acquire inputs and to manipulate them to produce the desired output in terms of functions and pe

In [20]:
chunks[0].metadata['section']

'The Term Software'

In [23]:
# Get chunks for topic
chunks_topic = get_topic_chunks(store, "Software Engineering Intro", "CSC 403")
chunks_topic

[Retriever] Topic sweep — fetching all chunks from store...
[Retriever] → 39 chunk(s) merged into existing sections (duplicate titles collapsed).
[Retriever] → 49 chunk(s) across 10 section(s).
[Retriever] → Topic map built: 10 section(s), all chunks retained.


{'Course Contents': [Document(id='doc_b87bdd53_0', metadata={'category': 'Title', 'topic': 'Software Engineering Intro', 'page_number': 1, 'doc_index': 0, 'section': 'Course Contents', 'content_length': 16, 'course': 'CSC 403', 'source': 'CSC 403_Software Engineering Intro_Software Engineering Intro', 'element_id': '330cf4934ef83395d3354a751d440c5e', 'condensed': False, 'filename': 'Software Engineering Intro.pdf'}, page_content='Course Contents:'),
  Document(id='doc_95c53b4b_1', metadata={'element_id': '178ec6be141992c8bbf48e59124c3669', 'source': 'CSC 403_Software Engineering Intro_Software Engineering Intro', 'doc_index': 1, 'page_number': 1, 'filename': 'Software Engineering Intro.pdf', 'section': 'Course Contents', 'category': 'NarrativeText', 'content_length': 327, 'course': 'CSC 403', 'topic': 'Software Engineering Intro', 'condensed': False}, page_content='Introduction; Software Design: Software architecture, Design patterns, OO analysis & Design, Design for re-use. Using API’

In [ ]:
chunks_topic[]

In [18]:
topic_chunk_result = [doc for doc_list in chunks_topic.values() for doc in doc_list]
topic_chunk_result
# for i, doc in enumerate(topic_chunk_result):
#     display(Markdown(f"\n\n{doc.page_content}\n"))

[Document(id='doc_8acba737_0', metadata={'content_length': 53, 'condensed': False, 'course': 'CSC 315', 'section': 'COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (WEEK 2', 'doc_index': 0, 'source': 'CSC 315_Interconnection Structures_Interconnection Structures', 'filename': 'Interconnection Structures.pdf', 'page_number': 1, 'topic': 'Interconnection Structures', 'category': 'Title', 'element_id': 'fc8d39a607237108cc91e2af962c85be'}, page_content='COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (WEEK 2)'),
 Document(id='doc_665dc0eb_2', metadata={'content_length': 42, 'filename': 'Interconnection Structures.pdf', 'topic': 'Interconnection Structures', 'section': 'Interconnection Structures', 'element_id': '1cb8263010cac730f3abb32370c847ac', 'category': 'ListItem', 'condensed': False, 'source': 'CSC 315_Interconnection Structures_Interconnection Structures', 'course': 'CSC 315', 'page_number': 1, 'doc_index': 2}, page_content='• Bus structures: Single bus, Multiple bus'),
 Document(id='doc_

In [12]:
from app.agent.rag.chains.notes_chain import run_notes_chain
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

#### Notes chain

In [25]:
# llm = ChatOllama(model="gemma3:1b")
load_dotenv()
api_key = os.environ.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", api_key=api_key)

In [46]:
# Case: A slow student
pdf_path = run_notes_chain(
      student_id    = "student_1019",
      topic_map     = chunks_topic,
      current_topic = "Interconnection Structures",
      weak_topics=[],
      strong_topics = [],
      course="CSC 315",
      learning_pace = "slow",
      llm           = llm,
      embedder      = embedder,
      store         = store,
  )

  [token_service] Tokens remaining for 'student_1019': 10,000,000
  [token_guard] Checking tokens — student: student_1019 | remaining: 10000000 | chain: run_notes_chain

[notes_chain] ═══ Starting notes generation ═══
[notes_chain] Student: student_1019 | Topic: Interconnection Structures | Pace: slow
[notes_chain] Sections to process: 17

[notes_chain] Processing section 1/17: 'COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (WEEK'


INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'Introduction to Computer Architecture & Organizati' condensed.

[notes_chain] Processing section 2/17: 'Interconnection Structures:'


INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'Interconnection Structures:' condensed.

[notes_chain] Processing section 3/17: 'Interconnection structures'


...

[notes_chain] ✓ Section 'Word Length' condensed.

[notes_chain] Processing section 16/17: 'INSTRUCTION SEQUENCING'


INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'INSTRUCTION SEQUENCING' condensed.

[notes_chain] Processing section 17/17: 'References'


INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'References' condensed.
[notes_chain] PDF built: c:\Users\USER\Documents\AI projects\DenseLess\data\notes\student_1019_Interconnection_Structures_20260421_171651.pdf

[notes_chain] ═══ Notes generation complete ═══
[notes_chain] PDF: c:\Users\USER\Documents\AI projects\DenseLess\data\notes\student_1019_Interconnection_Structures_20260421_171651.pdf
[notes_chain] Total tokens — in: 15,828 | out: 6,386 | total: 22,214
  [token_service] Deducted 22,214 tokens — student: 'student_1019' | chain: run_notes_chain | in: 15,828 | out: 6,386 | remaining: 9,977,786
  [token_guard] ✓ Tokens deducted — student: student_1019 | consumed: 22214 | remaining: 9977786


In [27]:
# Case: An avg student
pdf_path = run_notes_chain(
      student_id    = "student_1019",
      topic_map     = chunks_topic,
      current_topic = "Interconnection Structures",
      weak_topics=[],
      strong_topics = [],
      course="CSC 315",
      learning_pace = "average",
      llm           = llm,
      embedder      = embedder,
      store         = store,
  )

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


  [token_service] Tokens remaining for 'student_1019': 9,828,956
  [token_guard] Checking tokens — student: student_1019 | remaining: 9828956 | chain: run_notes_chain

[notes_chain] ═══ Starting notes generation ═══
[notes_chain] Mode: Gemini API
[notes_chain] Student: student_1019 | Topic: Interconnection Structures | Pace: average
[notes_chain] Sections to process: 15

[notes_chain] Processing section 1/15: 'COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (WEEK'


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (WEEK' condensed.
[notes_chain] ⏳ Throttle: sleeping 5s before next section.


INFO:google_genai.models:AFC is enabled with max remote calls: 10.


...

[notes_chain] ⚠ Attempt 2/3 failed for 'INSTRUCTION SEQUENCING': [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1016). Retrying...
[notes_chain] ✗ All 3 parse attempts exhausted for 'INSTRUCTION SEQUENCING'. Last error: [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1016)
[notes_chain] ⏳ Backing off 30s before rate-limit retry 3/3 for 'INSTRUCTION SEQUENCING'.


INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:google_genai.models:AFC is enabled with max remote calls: 10.


[notes_chain] ⚠ Attempt 1/3 failed for 'INSTRUCTION SEQUENCING': [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1016). Retrying...


INFO:google_genai.models:AFC is enabled with max remote calls: 10.


[notes_chain] ⚠ Attempt 2/3 failed for 'INSTRUCTION SEQUENCING': [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1016). Retrying...
[notes_chain] ✗ All 3 parse attempts exhausted for 'INSTRUCTION SEQUENCING'. Last error: [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1016)
[notes_chain] ✗ RPM retries exhausted at section 15/15. Building partial PDF (14 section(s) completed).
[notes_chain] ✗ Section 'INSTRUCTION SEQUENCING' failed after 2 attempts. Inserting placeholder.
[notes_chain] PDF built: c:\Users\USER\Documents\AI projects\DenseLess\data\notes\student_1019_Interconnection_Structures_20260428_225526.pdf

[notes_chain] ═══ Notes generation complete ═══
[notes_chain] PDF: c:\Users\USER\Documents\AI projects\DenseLess\data\notes\student_1019_Interconnection_Structures_20260428_225526.pdf
[notes_chain] Sections completed: 15/15
[notes_chain] Total tokens — in: 5,294 | out: 1,906 | total: 7,200
  [token_service] Dedu

In [26]:
# Case: A fast student
pdf_path = run_notes_chain(
      student_id    = "student_1019",
      topic_map     = chunks_topic,
      current_topic = "Interconnection Structures",
      weak_topics=[],
      strong_topics = [],
      course="CSC 315",
      learning_pace = "fast",
      llm           = llm,
      embedder      = embedder,
      store         = store,
  )

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


  [token_service] Tokens remaining for 'student_1019': 9,848,255
  [token_guard] Checking tokens — student: student_1019 | remaining: 9848255 | chain: run_notes_chain

[notes_chain] ═══ Starting notes generation ═══
[notes_chain] Mode: Gemini API
[notes_chain] Student: student_1019 | Topic: Interconnection Structures | Pace: fast
[notes_chain] Sections to process: 15

[notes_chain] Processing section 1/15: 'COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (WEEK'


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'COMPUTER ARCHITECTURE & ORGANIZATION LECTURE (WEEK' condensed.
[notes_chain] ⏳ Throttle: sleeping 5s before next section.


INFO:google_genai.models:AFC is enabled with max remote calls: 10.


...

[notes_chain] ✓ Section 'Word length' condensed.
[notes_chain] ⏳ Throttle: sleeping 5s before next section.


INFO:google_genai.models:AFC is enabled with max remote calls: 10.



[notes_chain] Processing section 15/15: 'INSTRUCTION SEQUENCING'


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'INSTRUCTION SEQUENCING' condensed.
[notes_chain] PDF built: c:\Users\USER\Documents\AI projects\DenseLess\data\notes\student_1019_Interconnection_Structures_20260428_224202.pdf

[notes_chain] ═══ Notes generation complete ═══
[notes_chain] PDF: c:\Users\USER\Documents\AI projects\DenseLess\data\notes\student_1019_Interconnection_Structures_20260428_224202.pdf
[notes_chain] Sections completed: 15/15
[notes_chain] Total tokens — in: 15,872 | out: 3,427 | total: 19,299
  [token_service] Deducted 19,299 tokens — student: 'student_1019' | chain: run_notes_chain | in: 15,872 | out: 3,427 | remaining: 9,828,956
  [token_guard] ✓ Tokens deducted — student: student_1019 | consumed: 19299 | remaining: 9828956


In [31]:
chunks

[Document(id='doc_3e2d2e0b_39', metadata={'filename': 'Interconnection Structures.pdf', 'category': 'Title', 'condensed': False, 'content_length': 44, 'element_id': 'c7e6f4f3dd36572c3b52fddc8485d92e', 'doc_index': 39, 'section': 'Complex Instruction Set Architecture (CISC', 'page_number': 4, 'topic': 'Interconnection Structures', 'source': 'CSC 315_Interconnection Structures_Interconnection Structures', 'course': 'CSC 315'}, page_content='Complex Instruction Set Architecture (CISC):'),
 Document(id='doc_7b133a0e_40', metadata={'topic': 'Interconnection Structures', 'section': 'Complex Instruction Set Architecture (CISC', 'source': 'CSC 315_Interconnection Structures_Interconnection Structures', 'element_id': '0ef2dedacc16e3c7b47770518325a156', 'content_length': 157, 'doc_index': 40, 'course': 'CSC 315', 'category': 'NarrativeText', 'page_number': 4, 'condensed': False, 'filename': 'Interconnection Structures.pdf'}, page_content='Making hardware complex in order accommodate very complex

### QA chain

In [ ]:
llm = ChatOllama(model="gemma3:1b")
# load_dotenv()
# api_key = os.environ.get('GOOGLE_API_KEY')
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", api_key=api_key)

In [31]:
from app.agent.rag.chains.qa_chain import run_qa_chain
import json

In [32]:
result1 = run_qa_chain("student_1019", "Instruction and instruction types", [], False, "Interconnection Structures", store, llm, "fast")
answer_dict = json.loads(result1.content)
answer      = answer_dict["answer"]

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


  [token_service] Tokens remaining for 'student_1019': 9,796,883
  [token_guard] Checking tokens — student: student_1019 | remaining: 9796883 | chain: run_qa_chain

[qa_chain] ═══ Starting QA chain ═══
[qa_chain] Mode: Local Ollama
[qa_chain] Student: student_1019 | Pace: fast | Source: original PDF
[qa_chain] Question: 'Instruction and instruction types'
[qa_chain] History turns in context: 0
[Retriever] Phase 1 — Semantic search (top_k=5, threshold=0.4): 'Instruction and instruction types'
Querying Chroma (top_k=5, threshold=0.4): 'Represent this sentence for searching relevant passages: Ins...'
  → Retrieved 5 chunk(s), 5 passed threshold.
[Retriever] → 5 seed chunk(s) passed threshold.
[Retriever] Phase 2 — Section context expansion...
[Retriever] Expanding context across 2 section(s): {'INSTRUCTION SEQUENCING', 'Instruction and Instruction Types'}
[Retriever]   → Section 'INSTRUCTION SEQUENCING': 24 chunk(s) fetched.
[Retriever]   → Section 'Instruction and Instruction Types': 6 c

INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 503 Service Unavailable"
INFO:google_genai._api_client:Retrying google.genai._api_client.BaseApiClient._request_once in 1.2 seconds as it raised ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"



[qa_chain] ═══ QA chain complete ═══
[qa_chain] Confidence: high | Sources: []
[qa_chain] Total tokens — in: 1,046 | out: 349 | total: 1,395
  [token_service] Deducted 1,395 tokens — student: 'student_1019' | chain: run_qa_chain | in: 1,046 | out: 349 | remaining: 9,795,488
  [token_guard] ✓ Tokens deducted — student: student_1019 | consumed: 1395 | remaining: 9795488


In [33]:
display(Markdown(answer))

The instruction set of a computer is the interface between software and hardware. Instructions dictate hardware actions based on specific circumstances. Key characteristics of instruction sets include:

*   **Length:** Fixed versus variable.
*   **Addressing Modes:** How operands are specified.
*   **Number of Operands:** The count of data items an instruction operates on.
*   **Types of Operations:** The specific actions supported.

Four primary types of operations are supported:

1.  **Data Transfer:** Moving data between memory and processor registers.
2.  **Arithmetic & Logic:** Operations performed on data.
3.  **Program Sequencing & Control:** Managing the flow of program execution.
4.  **I/O Transfers:** Input/Output operations.

Basic instruction types, classified by operand count, include:

*   **Three Address Instructions:** (Details not provided in the material).
*   **Two Address Instructions:** Example: B <–[A] + [B].
*   **One Address Instructions:** Operands are added to the accumulator, and the sum is stored back in the accumulator. Example: Add contents of A to accumulator & store sum back to accumulator.
*   **Zero Address Instructions:** Operands are stored in a push-down stack structure.

In [34]:
answer_dict

{'answer': 'The instruction set of a computer is the interface between software and hardware. Instructions dictate hardware actions based on specific circumstances. Key characteristics of instruction sets include:\n\n*   **Length:** Fixed versus variable.\n*   **Addressing Modes:** How operands are specified.\n*   **Number of Operands:** The count of data items an instruction operates on.\n*   **Types of Operations:** The specific actions supported.\n\nFour primary types of operations are supported:\n\n1.  **Data Transfer:** Moving data between memory and processor registers.\n2.  **Arithmetic & Logic:** Operations performed on data.\n3.  **Program Sequencing & Control:** Managing the flow of program execution.\n4.  **I/O Transfers:** Input/Output operations.\n\nBasic instruction types, classified by operand count, include:\n\n*   **Three Address Instructions:** (Details not provided in the material).\n*   **Two Address Instructions:** Example: B <–[A] + [B].\n*   **One Address Instruc

### Quizz chain

In [4]:
from app.services.quiz_router import handle_quiz_request
import os 
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

In [5]:
# llm = ChatOllama(model="gemma3:1b")
load_dotenv()
api_key = os.environ.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", api_key=api_key)

In [27]:
# 2. Time-travel testing (simulates a future date to test scheduled quizzes)
result = handle_quiz_request(
    student_id      = "student_1019",
    course          = "CSC 403",
    topic           = "Software Engineering Intro",
    store           = store,
    llm             = llm,
    simulated_today = "2026-05-04"  # Optional: ISO date string
)
if result["status"] == "ok":
    quiz = result["quiz"]       # dict — 10 questions
else:
    print(result["message"])    # e.g. "Read your notes first."

[quiz_router] Request — student: 'student_1019', course: 'CSC 403', topic: 'Software Engineering Intro'
[quiz_router] 🕒 DEV MODE: Simulating today as 2026-05-04
[quiz_router] Phase detected: 'schedule_pending'
[quiz_router] Phase 3 triggered — generating revision schedule.
[quiz_router] Revision schedule generated: ['2026-05-05', '2026-05-07', '2026-05-11', '2026-05-18']
[quiz_router] Profile saved: c:\Users\USER\Documents\AI projects\DenseLess\data\profiles\student_1019.json
Great work completing your post-test. Your revision schedule has been set. First revision: 2026-05-05.


In [25]:
# 2. Time-travel testing (simulates a future date to test scheduled quizzes)
result = handle_quiz_request(
    student_id      = "student_1019",
    course          = "CSC 403",
    topic           = "Software Engineering Intro",
    store           = store,
    llm             = llm,
    simulated_today = "2026-05-11"  # Optional: ISO date string
)
if result["status"] == "ok":
    quiz = result["quiz"]       # dict — 10 questions
else:
    print(result["message"])    # e.g. "Read your notes first."

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


[quiz_router] Request — student: 'student_1019', course: 'CSC 403', topic: 'Software Engineering Intro'
[quiz_router] 🕒 DEV MODE: Simulating today as 2026-05-11
[quiz_router] Phase detected: 'revision'
[quiz_router] Retrieval strategy — adaptive (3 section(s))
  [token_service] Tokens remaining for 'student_1019': 9,751,644
  [token_guard] Checking tokens — student: student_1019 | remaining: 9751644 | chain: run_quiz_chain
[quiz_chain] Starting — student: 'student_1019', course: 'CSC 403', topic: 'Software Engineering Intro'
[quiz_chain] Adaptive retrieval — targeting 3 weak section(s): ['OR', 'The key differences are', 'Importance of Software']
[quiz_chain] Section 'OR': 3 chunk(s) fetched.
[quiz_chain] Section 'The key differences are': 3 chunk(s) fetched.
[quiz_chain] Section 'Importance of Software': 7 chunk(s) fetched.
[quiz_chain] Retrieved 13 chunk(s) for topic 'Software Engineering Intro'.
[quiz_chain] Prompt built — chunks: 13, source pages: [4, 5]
[quiz_chain] LLM call — rate

INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[quiz_chain] JsonOutputParser succeeded.
[quiz_chain] Running per-question source mapping...
[Retriever] Phase 1 — Semantic search (top_k=3, threshold=0.4): 'What is software engineering defined as in terms of its appl'
Querying Chroma (top_k=3, threshold=0.4): 'Represent this sentence for searching relevant passages: Wha...'
  → Retrieved 3 chunk(s), 3 passed threshold.
[Retriever] → 3 seed chunk(s) passed threshold.
[Retriever] Phase 2 — Section context expansion...
[Retriever] Expanding context across 2 section(s): {'The Term Software Engineering', 'OR'}
[Retriever]   → Section 'The Term Software Engineering': 6 chunk(s) fetched.
[Retriever]   → Section 'OR': 3 chunk(s) fetched.
[Retriever] → Expansion complete: 3 seed → 9 unique chunk(s) after deduplication.
[Retriever] → Final result: 9 chunk(s) returned to chain.
[quiz_chain] Q1 mapped — pages: [3, 4], section: The Term Software Engineering
[Retriever] Phase 1 — Semantic search (top_k=3, threshold=0.4): 'What is a primary aim of 

### Eval chain

In [6]:
from app.agent.rag.chains.eval_chain import run_eval_chain
import os, json
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

In [12]:
# llm = ChatOllama(model="gemma3:1b")
load_dotenv()
api_key = os.environ.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-lite", api_key=api_key)

In [ ]:
response = run_eval_chain(
    student_id="student_1019",
    topic="Software Engineering Intro",
    quiz_phase="revision",
    quiz_path="../../data/quizzes/student_1019_software_engineering_intro_20260504_165955.json",
    llm=llm,
    simulated_date="2026-05-07"
)
print(response.content["questions"][0]["score"])
print(response.content["questions"][0]["explanation"])

  [token_service] Tokens remaining for 'student_1019': 9,730,401
  [token_guard] Checking tokens — student: student_1019 | remaining: 9730401 | chain: run_eval_chain
[eval_chain] Quiz loaded from ..\..\data\quizzes\student_1019_software_engineering_intro_20260504_165955.json
[eval_chain] Starting eval — student: 'student_1019' | topic: 'Software Engineering Intro' | phase: 'revision' | questions: 10
[eval_chain] Grading question 1/10...


INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent "HTTP/1.1 429 Too Many Requests"
INFO:google_genai._api_client:Retrying google.genai._api_client.BaseApiClient._request_once in 1.51 seconds as it raised ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.goo